## Fix formatting issues

In [2]:
import csv
import io
import os
import re
import requests
from datetime import datetime

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------

SHEET_URL = "https://docs.google.com/spreadsheets/d/1lHUX4W8M2yKQSymul4Xu55jUri9fzgK2h7lHmQdPQoQ/export?format=csv"

BASE_DIR = "complaints"
MASTER_CSV = os.path.join(BASE_DIR, "master.csv")
REGISTRY_CSV = os.path.join(BASE_DIR, "image_registry.csv")
PROPERTY_CSV = os.path.join(BASE_DIR, "properties.csv")

os.makedirs(BASE_DIR, exist_ok=True)

# ------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------

def safe_slug(text):
    text = text.lower().strip()
    text = re.sub(r"[^\w\s-]", "", text)
    text = re.sub(r"\s+", "_", text)
    return text or "none"

def clean_text(value):
    if not value:
        return ""
    val_clean = re.sub(r"[\x00-\x1f\x7f-\x9f]", " ", str(value))
    return " ".join(val_clean.split()).strip()

def sanitize_multiline(value):
    if not value:
        return value
    return value.replace("\r\n", "\\n").replace("\n", "\\n")

def flatten_youtube_links(value):
    """Strip and remove all newlines from YouTube/links fields so they stay on a single line."""
    if not value:
        return ""
    flattened = re.sub(r"[\r\n]+", "; ", str(value))
    return " ".join(flattened.split()).strip()

def is_youtube_url(url):
    if not url:
        return False
    lower_url = url.lower()
    return "youtube.com" in lower_url or "youtu.be" in lower_url

def extract_drive_id(url):
    if "id=" in url:
        return url.split("id=")[1]
    m = re.search(r"/d/([^/]+)", url)
    if m:
        return m.group(1)
    return None

def download_drive_file(file_id, dest_path):
    url = f"https://drive.google.com/uc?export=download&id={file_id}"
    r = requests.get(url, stream=True)
    r.raise_for_status()
    with open(dest_path, "wb") as f:
        for chunk in r.iter_content(8192):
            f.write(chunk)
    return dest_path

def next_available_filename(base_path):
    if not os.path.exists(base_path):
        return base_path
    root, ext = os.path.splitext(base_path)
    counter = 1
    while True:
        candidate = f"{root}_{counter}{ext}"
        if not os.path.exists(candidate):
            return candidate
        counter += 1

# ------------------------------------------------------------
# LOAD OR CREATE IMAGE REGISTRY
# ------------------------------------------------------------

registry = {}

if os.path.exists(REGISTRY_CSV):
    with open(REGISTRY_CSV, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            registry[row["file_id"]] = row["saved_path"]
else:
    with open(REGISTRY_CSV, "w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["file_id", "saved_path"])
        writer.writeheader()

# ------------------------------------------------------------
# LOAD EXISTING PROPERTIES (PREVENT DATA LOSS & DUPLICATES)
# ------------------------------------------------------------

properties = {}  # address.lower() → property record

for target_csv in [MASTER_CSV, PROPERTY_CSV]:
    if os.path.exists(target_csv):
        with open(target_csv, "r", encoding="utf-8-sig", newline="") as f:
            reader = csv.DictReader(f)
            if reader.fieldnames and "Nature of Complaint" in reader.fieldnames:
                continue
            for row in reader:
                addr_raw = clean_text(row.get("Address", ""))
                if not addr_raw:
                    continue
                key = addr_raw.lower()
                
                other_raw = row.get("Other Listing Site", "")
                other_urls = []
                for p in re.split(r"[;,\s]+", other_raw):
                    p_clean = clean_text(p)
                    if p_clean and not is_youtube_url(p_clean):
                        if p_clean not in other_urls:
                            other_urls.append(p_clean)

                airbnb = clean_text(row.get("Airbnb URL", ""))
                vrbo = clean_text(row.get("VRBO URL", ""))

                properties[key] = {
                    "Address": addr_raw,
                    "Latitude": clean_text(row.get("Latitude", "")),
                    "Longitude": clean_text(row.get("Longitude", "")),
                    "Airbnb URL": "" if is_youtube_url(airbnb) else airbnb,
                    "VRBO URL": "" if is_youtube_url(vrbo) else vrbo,
                    "Other Listing Site": other_urls,
                    "Permit Number": clean_text(row.get("Permit Number", "")),
                    "Bedrooms": clean_text(row.get("Bedrooms", "")),
                    "Guests": clean_text(row.get("Guests", "")),
                    "Parking Spaces": clean_text(row.get("Parking Spaces", "")),
                    "Agent": clean_text(row.get("Agent", ""))
                }

# ------------------------------------------------------------
# STEP 1 — FETCH & PARSE SHEET DATA IN MEMORY
# ------------------------------------------------------------

print("[+] Fetching sheet data...")
response = requests.get(SHEET_URL)
response.raise_for_status()

f_in = io.StringIO(response.text)
reader = csv.DictReader(f_in)
fieldnames_raw = reader.fieldnames

print("[+] Processing rows...")

for idx, row in enumerate(reader, start=1):

    address_raw = clean_text(row.get("Address of Rental Property", ""))
    if not address_raw or address_raw.lower() == "unknown":
        continue

    unit_raw = clean_text(row.get("Unit/Apartment Number", "none"))
    address = safe_slug(address_raw)
    unit = safe_slug(unit_raw)

    nature = safe_slug(row.get("Nature of Complaint", "unknown"))
    date_incident = row.get("Date of Incident", "").strip()
    timestamp = row.get("Timestamp", "").strip()

    try:
        parsed_date = datetime.strptime(date_incident, "%m/%d/%Y")
        date_slug = parsed_date.strftime("%Y-%m-%d")
    except:
        date_slug = "unknown_date"

    addr_dir = os.path.join(BASE_DIR, address)
    unit_dir = os.path.join(addr_dir, unit)
    os.makedirs(unit_dir, exist_ok=True)

    # ------------------------------------------------------------
    # EXTRACT PROPERTY DETAILS (NO YOUTUBE, NO COMPLAINTS)
    # ------------------------------------------------------------

    lat = clean_text(row.get("Latitude (Place marker on map to autofill)", ""))
    lng = clean_text(row.get("Longitude (Place marker on map to autofill)", ""))
    
    airbnb = clean_text(row.get("Airbnb URL", ""))
    if is_youtube_url(airbnb):
        airbnb = ""

    vrbo = clean_text(row.get("VRBO URL", ""))
    if is_youtube_url(vrbo):
        vrbo = ""

    other_raw = clean_text(row.get("Other Listing Site", ""))
    other_urls = []
    if other_raw:
        parts = re.split(r"[,\s;]+", other_raw)
        for p in parts:
            p_clean = clean_text(p)
            if p_clean and not is_youtube_url(p_clean):
                if p_clean not in other_urls:
                    other_urls.append(p_clean)

    key = address_raw.lower()

    if key not in properties:
        properties[key] = {
            "Address": address_raw,
            "Latitude": lat,
            "Longitude": lng,
            "Airbnb URL": airbnb,
            "VRBO URL": vrbo,
            "Other Listing Site": other_urls,
            "Permit Number": "",
            "Bedrooms": "",
            "Guests": "",
            "Parking Spaces": "",
            "Agent": ""
        }
    else:
        if lat and not properties[key]["Latitude"]:
            properties[key]["Latitude"] = lat
        if lng and not properties[key]["Longitude"]:
            properties[key]["Longitude"] = lng
        if airbnb and not properties[key]["Airbnb URL"]:
            properties[key]["Airbnb URL"] = airbnb
        if vrbo and not properties[key]["VRBO URL"]:
            properties[key]["VRBO URL"] = vrbo

        existing = properties[key]["Other Listing Site"]
        for u in other_urls:
            if u not in existing:
                existing.append(u)

    # ------------------------------------------------------------
    # PER-UNIT COMPLAINT CSV (FLATTENS YOUTUBE URL NEWLINES)
    # ------------------------------------------------------------

    unit_csv = os.path.join(unit_dir, "complaints.csv")
    new_file = not os.path.exists(unit_csv)

    existing_keys = set()
    if not new_file:
        with open(unit_csv, "r", encoding="utf-8") as uf:
            existing_reader = csv.DictReader(uf)
            for existing_row in existing_reader:
                row_key_tuple = (
                    existing_row.get("Timestamp", "").strip(),
                    existing_row.get("Address of Rental Property", "").strip(),
                    existing_row.get("Unit/Apartment Number", "").strip()
                )
                existing_keys.add(row_key_tuple)

    # Flatten newlines in any link/YouTube fields so they stay on a single line
    processed_row = {}
    for k, v in row.items():
        if "Links" in k or "YouTube" in k:
            processed_row[k] = flatten_youtube_links(v)
        else:
            processed_row[k] = sanitize_multiline(v)

    row_key = (timestamp, address_raw, unit_raw)

    if row_key in existing_keys:
        print(f"    ✔ Row {idx}: Duplicate complaint skipped.")
    else:
        with open(unit_csv, "a", encoding="utf-8", newline="") as uf:
            writer = csv.DictWriter(uf, fieldnames=fieldnames_raw)
            if new_file:
                writer.writeheader()
            writer.writerow(processed_row)
            print(f"    ✔ Row {idx}: Complaint logged.")

    # ------------------------------------------------------------
    # MULTI-IMAGE SUPPORT
    # ------------------------------------------------------------

    img_field = row.get("Upload Images or Documents", "").strip()
    if img_field:
        raw_urls = re.split(r"[,\s]+", img_field)
        urls = [u.strip() for u in raw_urls if u.strip()]

        for img_url in urls:
            file_id = extract_drive_id(img_url)
            if not file_id:
                continue

            if file_id in registry:
                continue

            base_filename = f"{nature}_{date_slug}.jpg"
            base_path = os.path.join(unit_dir, base_filename)
            final_path = next_available_filename(base_path)

            try:
                download_drive_file(file_id, final_path)
                registry[file_id] = final_path
                with open(REGISTRY_CSV, "a", encoding="utf-8", newline="") as rf:
                    writer = csv.DictWriter(rf, fieldnames=["file_id", "saved_path"])
                    writer.writerow({"file_id": file_id, "saved_path": final_path})
            except Exception as e:
                print(f"    ✖ Row {idx}: Failed to download image {img_url}: {e}")

print("[+] Ingestion complete.")

# ------------------------------------------------------------
# STEP 2 — WRITE CLEAN MASTER & PROPERTY CSVs (NO COMPLAINTS / NO YOUTUBE)
# ------------------------------------------------------------

fieldnames = [
    "Address",
    "Latitude",
    "Longitude",
    "Airbnb URL",
    "VRBO URL",
    "Other Listing Site",
    "Permit Number",
    "Bedrooms",
    "Guests",
    "Parking Spaces",
    "Agent"
]

for target_csv in [MASTER_CSV, PROPERTY_CSV]:
    with open(target_csv, "w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()

        for prop in properties.values():
            other_joined = "; ".join(prop["Other Listing Site"])
            row_out = prop.copy()
            row_out["Other Listing Site"] = other_joined
            writer.writerow(row_out)

print("[+] Clean master & property CSVs updated successfully.")
print("[+] Done!")

[+] Fetching sheet data...
[+] Processing rows...
    ✔ Row 1: Duplicate complaint skipped.
    ✔ Row 2: Duplicate complaint skipped.
    ✔ Row 3: Duplicate complaint skipped.
    ✔ Row 4: Duplicate complaint skipped.
[+] Ingestion complete.
[+] Clean master & property CSVs updated successfully.
[+] Done!
